# IndexCalc — Phase 6d 테스트

Component Evaluation: `evaluate(expr, components)` → numeric array via einsum.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from indexcalc import (
    IndexSpace, Tensor, IndexRegistry, parse, to_latex, evaluate,
    trace, Trace, expand_covariant, covariant, LeviCivitaConnection,
    partial, expand_partial,
)
from IPython.display import display, Math

In [2]:
spacetime = IndexSpace("spacetime", dim=4, indices="μνλρσ", metric="g")
lorentz   = IndexSpace("lorentz",   dim=4, indices="abcde", metric="η")

reg = IndexRegistry()
reg.register(spacetime)
reg.register(lorentz)

eta = np.diag([-1., 1., 1., 1.])

## 1. Index lowering: $\eta_{ab} V^b \to V_a$

In [3]:
V_arr = np.array([2., 1., 0., -1.])

eta_T = Tensor("η", [lorentz.lower("a"), lorentz.lower("b")])
V_T   = Tensor("V", [lorentz.upper("b")])

expr1 = eta_T * V_T
result1 = evaluate(expr1, {"η": eta, "V": V_arr})

display(Math(to_latex(expr1) + r" \;=\; " + str(result1)))
print(f"기대: [-2, 1, 0, -1]")

<IPython.core.display.Math object>

기대: [-2, 1, 0, -1]


## 2. 행렬 곱: $T^i{}_j S^j{}_k$

In [4]:
small = IndexSpace("small", dim=2, indices="ijk", metric="δ")

T_arr = np.array([[1, 2], [3, 4]], dtype=float)
S_arr = np.array([[5, 6], [7, 8]], dtype=float)

T_t = Tensor("T", [small.upper("i"), small.lower("j")])
S_t = Tensor("S", [small.upper("j"), small.lower("k")])

prod = T_t * S_t
result2 = evaluate(prod, {"T": T_arr, "S": S_arr})

print(f"T·S =\n{result2}")
print(f"np.matmul =\n{T_arr @ S_arr}")
assert np.allclose(result2, T_arr @ S_arr), "FAIL"
print("✓ 일치")

T·S =
[[19. 22.]
 [43. 50.]]
np.matmul =
[[19. 22.]
 [43. 50.]]
✓ 일치


## 3. Trace: $T^i{}_i$

In [5]:
T_trace = Tensor("T", [small.upper("i"), small.lower("i")])
tr = trace(T_trace, "i")

result3 = evaluate(tr, {"T": T_arr})
print(f"Tr(T) = {result3},  np.trace = {np.trace(T_arr)}")
assert np.isclose(result3, np.trace(T_arr)), "FAIL"
print("✓ 일치")

Tr(T) = 5.0,  np.trace = 5.0
✓ 일치


## 4. ScalarMul & TensorSum: $2A + B$

In [6]:
A_t = Tensor("A", [small.upper("i"), small.lower("j")])
B_t = Tensor("B", [small.upper("i"), small.lower("j")])

A_arr = np.eye(2)
B_arr = np.ones((2, 2))

expr4 = 2 * A_t + B_t
result4 = evaluate(expr4, {"A": A_arr, "B": B_arr})
expected4 = 2*A_arr + B_arr

print(f"2A + B =\n{result4}")
assert np.allclose(result4, expected4), "FAIL"
print("✓ 일치")

2A + B =
[[3. 1.]
 [1. 3.]]
✓ 일치


## 5. Metric 구분: $g_{\mu\nu} g^{\nu\lambda} = \delta^\lambda{}_\mu$

같은 이름 "g"의 metric과 inverse를 `("g","dd")`, `("g","uu")` 키로 구분.

In [7]:
metric = np.diag([-1., 1., 1., 1.])
inv_metric = np.diag([-1., 1., 1., 1.])

g_lower = Tensor("g", [spacetime.lower("μ"), spacetime.lower("ν")])
g_upper = Tensor("g", [spacetime.upper("ν"), spacetime.upper("λ")])

prod5 = g_lower * g_upper
result5 = evaluate(prod5, {("g", "dd"): metric, ("g", "uu"): inv_metric})

display(Math(to_latex(prod5) + r" \;=\; \delta^\lambda{}_\mu"))
print(f"결과:\n{result5}")
assert np.allclose(result5, np.eye(4)), "FAIL"
print("✓ identity 확인")

<IPython.core.display.Math object>

결과:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
✓ identity 확인


## 6. 전개된 공변미분: $\nabla_\mu V^\nu = \partial_\mu V^\nu + \Gamma^\nu{}_{\mu\rho} V^\rho$

`expand_covariant()` 후 `evaluate()`로 숫자 계산. 평탄 시공간에서 $\Gamma=0$이면 $\nabla = \partial$.

In [8]:
g     = Tensor("g", [spacetime.lower("μ"), spacetime.lower("ν")])
g_inv = Tensor("g", [spacetime.upper("μ"), spacetime.upper("ν")])
christoffel = LeviCivitaConnection(g, g_inv, spacetime)

V_up = Tensor("V", [spacetime.upper("ν")])
mu = spacetime.lower("μ")

nabla_V = covariant(V_up, mu, christoffel)
expanded = expand_covariant(nabla_V)

display(Math(r"\nabla_\mu V^\nu \;=\; " + to_latex(expanded)))

<IPython.core.display.Math object>

In [9]:
# 평탄 시공간: Γ = 0
V_arr = np.array([1., 0., 0., 0.])
dV_arr = np.zeros((4, 4))
dV_arr[1, 0] = 0.5  # ∂_1 V^0 = 0.5
dV_arr[2, 3] = -1.0  # ∂_2 V^3 = -1.0
Gamma_zero = np.zeros((4, 4, 4))

result6 = evaluate(expanded, {"∂V": dV_arr, "Γ": Gamma_zero, "V": V_arr})
print("Flat spacetime (Γ=0):")
print(f"∇_μ V^ν =\n{result6}")
print(f"\n∂_μ V^ν =\n{dV_arr}")
assert np.allclose(result6, dV_arr), "FAIL"
print("✓ Γ=0일 때 ∇ = ∂ 확인")

Flat spacetime (Γ=0):
∇_μ V^ν =
[[ 0.   0.   0.   0. ]
 [ 0.5  0.   0.   0. ]
 [ 0.   0.   0.  -1. ]
 [ 0.   0.   0.   0. ]]

∂_μ V^ν =
[[ 0.   0.   0.   0. ]
 [ 0.5  0.   0.   0. ]
 [ 0.   0.   0.  -1. ]
 [ 0.   0.   0.   0. ]]
✓ Γ=0일 때 ∇ = ∂ 확인


## 7. Nonzero Γ: Schwarzschild-like Christoffel

$\Gamma \neq 0$인 경우 $\nabla_\mu V^\nu \neq \partial_\mu V^\nu$ 확인.

In [10]:
# 간단한 Γ 예시: Γ^0_{10} = 0.3 (하나만 nonzero)
Gamma = np.zeros((4, 4, 4))
Gamma[0, 1, 0] = 0.3  # Γ^0_{1,0}

# ∇_μ V^ν = ∂_μ V^ν + Γ^ν_{μρ} V^ρ
# Γ^ν_{μρ} V^ρ 항: Γ^0_{1,0} * V^0 = 0.3 * 1.0 = 0.3 → (μ=1, ν=0) 성분에 추가
expected7 = dV_arr.copy()
expected7[1, 0] += 0.3  # Γ contribution

result7 = evaluate(expanded, {"∂V": dV_arr, "Γ": Gamma, "V": V_arr})
print(f"∇_μ V^ν (Γ≠0) =\n{result7}")
print(f"\n기대값 =\n{expected7}")
assert np.allclose(result7, expected7), "FAIL"
print("✓ Γ 기여분 확인")

∇_μ V^ν (Γ≠0) =
[[ 0.   0.   0.   0. ]
 [ 0.8  0.   0.   0. ]
 [ 0.   0.   0.  -1. ]
 [ 0.   0.   0.   0. ]]

기대값 =
[[ 0.   0.   0.   0. ]
 [ 0.8  0.   0.   0. ]
 [ 0.   0.   0.  -1. ]
 [ 0.   0.   0.   0. ]]
✓ Γ 기여분 확인


## 8. 3-tensor contraction: $R^{\mu}{}_{\nu\rho\sigma} V^{\rho}$

고차 텐서의 부분 축약.

In [11]:
# 간단한 (1,3) Riemann-like 텐서 (랜덤)
np.random.seed(42)
R_arr = np.random.randn(4, 4, 4, 4).round(2)
V_4 = np.array([1., 0.5, -0.3, 0.])

R_t = Tensor("R", [
    spacetime.upper("μ"),
    spacetime.lower("ν"),
    spacetime.lower("ρ"),
    spacetime.lower("σ"),
])
V_t = Tensor("V", [spacetime.upper("ρ")])

expr8 = R_t * V_t  # contract on ρ
result8 = evaluate(expr8, {"R": R_arr, "V": V_4})

# 수동 검증: np.einsum("abcd,c->abd", R, V)
expected8 = np.einsum("abcd,c->abd", R_arr, V_4)

print(f"R^μ_{{νρσ}} V^ρ: shape = {result8.shape}")
assert np.allclose(result8, expected8), "FAIL"
print("✓ 고차 텐서 contraction 일치")

R^μ_{νρσ} V^ρ: shape = (4, 4, 4)
✓ 고차 텐서 contraction 일치
